In [7]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import TimeoutException, NoSuchElementException, StaleElementReferenceException, WebDriverException
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.action_chains import ActionChains
import time
import pandas as pd

In [2]:
# Create options object
options = Options()

#options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--disable-infobars')
options.add_argument('--disable-extensions')
options.add_argument('--disable-notifications')
options.add_argument('--window-size=1724,968')
user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/96.0.4664.110 Safari/537.36"
options.add_argument(f'user-agent={user_agent}')

In [3]:
vivino_page = "https://www.vivino.com/explore?e=eJwlzDsKgDAQBNDbTB0_7RaCV7ASkTVGCZhEkvi7vYs284oZxkUq4KwnBcc3FUop6IeaDlqixS71utDJ0ZrMG0KcaTZJI0wPRc7Wr2nU4fAZV-4HWX9UP-VPLZdJ5ANfvskDJxc%3D"

In [4]:
def scroll_down(element):
    #delta_y = element.rect['y']
    ActionChains(driver)\
        .scroll_by_amount(0, 100)\
        .perform()

def scroll_to_element(element):
    ActionChains(driver)\
        .scroll_to_element(element)\
        .perform()

def expand_slider(driver):
    # Locate the range slider container
    slider_container = driver.find_element(By.CLASS_NAME, "rangeSlider__slider--DaZc5")

    # Locate the two slider handles within the container
    handles = slider_container.find_elements(By.CLASS_NAME, 'rc-slider-handle')

    if len(handles) != 2:
        raise ValueError("Expected exactly 2 handles, found: {}".format(len(handles)))

    handle1 = handles[0]
    handle2 = handles[1]

    # Initialize ActionChains
    actions = ActionChains(driver)

    # Get the width of the slider to calculate the movement
    slider = slider_container.find_element(By.CLASS_NAME, 'rc-slider')
    slider_width = slider.size['width']

    # Calculate the offset to move handle1 to the far left
    handle1_x_offset = handle1.location['x'] - slider.location['x']
    # Calculate the offset to move handle2 to the far right
    handle2_x_offset = slider_width - (handle2.location['x'] - slider.location['x'])

    # Drag handle1 to the far left (minimum value)
    actions.click_and_hold(handle1).move_by_offset(-handle1_x_offset, 0).release().perform()

    # Drag handle2 to the far right (maximum value)
    actions.click_and_hold(handle2).move_by_offset(handle2_x_offset, 0).release().perform()

def select_ship_to_country(driver, country):
    try:
        # Locate the "Ship to" dropdown container by checking for the "Ship to" text
        dropdown_container = driver.find_element(By.XPATH, "//div[contains(@class, 'navigationItem_navigationItem__Owbn-')]//span[text()='Ship to']/ancestor::div[contains(@class, 'navigationItem_labelContainer__aZ5-F')]")
        
        # Click the dropdown container to reveal options
        dropdown_container.click()

        # Wait for the dropdown options to be visible
        WebDriverWait(driver, 10).until(
            EC.visibility_of_element_located((By.XPATH, "//div[contains(@class, 'navigationItem_closeWrapper__n28xG')]"))
        )

        # Locate the option for the desired country and click it
        country_option = driver.find_element(By.XPATH, f"//a[@data-shiptotype='Country' and text()='{country}']")
        country_option.click()

        print(f"Successfully selected {country} from the dropdown.")
        time.sleep(4)

    except Exception as e:
        print(f"An error occurred: {e}")

number_to_retrieve = 200
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
wine_hrefs = []
wine_brand_names = []
wine_product_names = []
driver.get(vivino_page)

# Select "Australia" as ship to location
select_ship_to_country(driver, "Australia")
                      
#expand price slider - skipping this as slider seems to update automatically whenever country is changed to australia
#expand_slider(driver)

wine_cards = driver.find_elements(By.CLASS_NAME, "card__card--2R5Wh")
foot_header = driver.find_element(By.CLASS_NAME, "footer_header__cq48I")
n_wine_cards = len(wine_cards)
previous_last_result = wine_cards[-1].text
while n_wine_cards < number_to_retrieve:
    print(f"{n_wine_cards} wine results found. Continuing to scroll...")
    scroll_to_element(foot_header)

    # check results have updated
    while True:
        wine_cards = driver.find_elements(By.CLASS_NAME, "card__card--2R5Wh")
        current_last_result = wine_cards[-1].text
        if current_last_result == previous_last_result:
            print("Results not updated. Waiting and retrieving again...")
            time.sleep(1)
            continue
        else:
            n_wine_cards = len(wine_cards)
            previous_last_result = current_last_result
            break
       
print(f"{n_wine_cards} wine results found. Stopping scrolling and retrieving results...")


Successfully selected Australia from the dropdown.
25 wine results found. Continuing to scroll...
Results not updated. Waiting and retrieving again...
50 wine results found. Continuing to scroll...
75 wine results found. Continuing to scroll...
Results not updated. Waiting and retrieving again...
Results not updated. Waiting and retrieving again...
Results not updated. Waiting and retrieving again...
Results not updated. Waiting and retrieving again...
Results not updated. Waiting and retrieving again...
Results not updated. Waiting and retrieving again...
Results not updated. Waiting and retrieving again...
Results not updated. Waiting and retrieving again...
Results not updated. Waiting and retrieving again...
Results not updated. Waiting and retrieving again...
Results not updated. Waiting and retrieving again...
Results not updated. Waiting and retrieving again...
Results not updated. Waiting and retrieving again...
Results not updated. Waiting and retrieving again...
Results not u

In [8]:
product_hrefs = []
brand_names = []
product_names = []
for i in range(len(wine_cards)):
    print(f"Retriving data from result {i} of {n_wine_cards}")
    product_anchor = wine_cards[i].find_element(By.CLASS_NAME, "anchor_anchor__m8Qi-")
    product_hrefs.append(product_anchor.get_attribute("href"))
    brand_and_product_names = wine_cards[i].find_elements(By.CLASS_NAME, "wineInfoVintage__truncate--3QAtw")
    brand_names.append(brand_and_product_names[0].text)
    product_names.append(brand_and_product_names[1].text)
    time.sleep(0.1)

vivino_product_hrefs_df = pd.DataFrame({
    "product_href" : product_hrefs,
    "brand_name" : brand_names ,
    "product_name" : product_names,
})

In [10]:
vivino_product_hrefs_df

,product_href,brand_name,product_name
0,https://www.vivino.com/AU/en/nineteen-crimes-r...,19 Crimes,Red Blend
1,https://www.vivino.com/AU/en/penfolds-bin-389-...,Penfolds,Bin 389 Cabernet - Shiraz 1985
2,https://www.vivino.com/AU/en/saltram-pepperjac...,Saltram,Pepperjack Shiraz 2022
3,https://www.vivino.com/AU/en/two-hands-sexy-be...,Two Hands,Sexy Beast Cabernet Sauvignon 2023
4,https://www.vivino.com/AU/en/penfolds-rwt-shir...,Penfolds,RWT Shiraz 2020
...,...,...,...
195,https://www.vivino.com/AU/en/pooley-pinot-noir...,Pooley,Pinot Noir 2023
196,https://www.vivino.com/AU/en/eddystone-point-p...,Eddystone Point,Pinot Noir 2022
197,https://www.vivino.com/AU/en/d-arenberg-the-pe...,d'Arenberg,The Peppermint Paddock Chambourcin N.V.
198,https://www.vivino.com/AU/en/kooyong-estate-pi...,Kooyong,Estate Pinot Noir 2022


In [12]:
file_path = f"../data/vivino/popular_vivino_hrefs.csv"
vivino_product_hrefs_df.to_csv(file_path, index=False)